In [1]:
import sys
print(sys.version)
import pandas
print(pandas.__version__)

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
3.1.1


In [2]:
import pandas as pd

RAW = "../data/raw/"

customers   = pd.read_csv(RAW + "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW + "olist_geolocation_dataset.csv")
orders      = pd.read_csv(RAW + "olist_orders_dataset.csv")
items       = pd.read_csv(RAW + "olist_order_items_dataset.csv")
payments    = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
products    = pd.read_csv(RAW + "olist_products_dataset.csv")
sellers     = pd.read_csv(RAW + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(RAW + "product_category_name_translation.csv")

tables = {
    "customers": customers, "geolocation": geolocation, "orders": orders,
    "items": items, "payments": payments, "reviews": reviews,
    "products": products, "sellers": sellers,
    "category_translation": category_translation,
}

for name, df in tables.items():
    print(f"\n===== {name} {df.shape} =====")
    print(list(df.columns))


===== customers (99441, 5) =====
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

===== geolocation (1000163, 5) =====
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

===== orders (99441, 8) =====
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

===== items (112650, 7) =====
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

===== payments (103886, 5) =====
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

===== reviews (99224, 7) =====
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

===== products (32951, 9) =====
['product_id', 'product

## Data Dictionary

The SQLite database has 9 base tables (cleaned source data) and 4 engineered feature tables (notebook 05).

## Base tables

### `customers`

Customers and their location.

| Column | Type | Key | Description |
|---|---|---|---|
| customer_id | TEXT | PK | Customer ID for one order (a new ID is issued per order) |
| customer_unique_id | TEXT |  | Unique ID of the real customer across orders |
| customer_zip_code_prefix | TEXT |  | First digits of the customer zip code |
| customer_city | TEXT |  | Customer city |
| customer_state | TEXT |  | Customer state (2-letter code) |

### `geolocation`

Zip code prefix coordinates (one row per prefix after cleaning).

| Column | Type | Key | Description |
|---|---|---|---|
| geolocation_zip_code_prefix | TEXT | PK | Zip code prefix (one row per prefix after cleaning) |
| geolocation_lat | REAL |  | Latitude |
| geolocation_lng | REAL |  | Longitude |
| geolocation_city | TEXT |  | City of the zip prefix |
| geolocation_state | TEXT |  | State of the zip prefix |

### `orders`

One row per order with status and delivery dates.

| Column | Type | Key | Description |
|---|---|---|---|
| order_id | TEXT | PK | Unique order ID |
| customer_id | TEXT | FK -> customers.customer_id | Customer ID for one order (a new ID is issued per order) |
| order_status | TEXT |  | Order status (delivered, shipped, canceled, ...) |
| order_purchase_timestamp | TIMESTAMP |  | When the order was placed |
| order_approved_at | TIMESTAMP |  | When the payment was approved |
| order_delivered_carrier_date | TIMESTAMP |  | When the order was handed to the carrier |
| order_delivered_customer_date | TIMESTAMP |  | When the customer received the order |
| order_estimated_delivery_date | TIMESTAMP |  | Estimated delivery date shown to the customer |

### `items`

Items inside each order (links orders, products and sellers).

| Column | Type | Key | Description |
|---|---|---|---|
| order_id | TEXT | PK, FK -> orders.order_id | Unique order ID |
| order_item_id | INTEGER | PK | Item sequence number within the order |
| product_id | TEXT | FK -> products.product_id | Unique product ID |
| seller_id | TEXT | FK -> sellers.seller_id | Unique seller ID |
| shipping_limit_date | TIMESTAMP |  | Deadline for the seller to hand the item to the carrier |
| price | REAL |  | Item price (BRL) |
| freight_value | REAL |  | Freight cost for the item (BRL) |

### `payments`

Payments for each order.

| Column | Type | Key | Description |
|---|---|---|---|
| order_id | TEXT | PK, FK -> orders.order_id | Unique order ID |
| payment_sequential | INTEGER | PK | Sequence number when an order uses several payments |
| payment_type | TEXT |  | Payment method (credit_card, boleto, voucher, debit_card) |
| payment_installments | INTEGER |  | Number of installments chosen |
| payment_value | REAL |  | Value of the payment (BRL) |

### `reviews`

Customer review for each order (one per order after cleaning).

| Column | Type | Key | Description |
|---|---|---|---|
| review_id | TEXT | PK | Unique review ID |
| order_id | TEXT | FK -> orders.order_id | Unique order ID |
| review_score | INTEGER |  | Customer rating from 1 to 5 |
| review_comment_title | TEXT |  | Review title (optional) |
| review_comment_message | TEXT |  | Review text (optional) |
| review_creation_date | TIMESTAMP |  | When the review survey was sent |
| review_answer_timestamp | TIMESTAMP |  | When the customer answered the survey |

### `products`

Product attributes and category.

| Column | Type | Key | Description |
|---|---|---|---|
| product_id | TEXT | PK | Unique product ID |
| product_category_name | TEXT | FK -> category_translation.product_category_name | Product category (Portuguese) |
| product_name_length | REAL |  | Characters in the product name |
| product_description_length | REAL |  | Characters in the product description |
| product_photos_qty | REAL |  | Number of product photos |
| product_weight_g | REAL |  | Product weight in grams |
| product_length_cm | REAL |  | Product length in cm |
| product_height_cm | REAL |  | Product height in cm |
| product_width_cm | REAL |  | Product width in cm |

### `sellers`

Sellers and their location.

| Column | Type | Key | Description |
|---|---|---|---|
| seller_id | TEXT | PK | Unique seller ID |
| seller_zip_code_prefix | TEXT |  | First digits of the seller zip code |
| seller_city | TEXT |  | Seller city |
| seller_state | TEXT |  | Seller state (2-letter code) |

### `category_translation`

Portuguese to English product category names.

| Column | Type | Key | Description |
|---|---|---|---|
| product_category_name | TEXT | PK | Product category (Portuguese) |
| product_category_name_english | TEXT |  | Product category (English) |

## Feature tables

These tables are derived from the base tables and have no keys of their own.

### `order_features`

Engineered order-level features (notebook 05).

| Column | Type | Key | Description |
|---|---|---|---|
| order_id | TEXT |  | Unique order ID |
| customer_id | TEXT |  | Customer ID for one order (a new ID is issued per order) |
| order_status | TEXT |  | Order status (delivered, shipped, canceled, ...) |
| order_purchase_timestamp | DATETIME |  | When the order was placed |
| order_approved_at | DATETIME |  | When the payment was approved |
| order_delivered_carrier_date | DATETIME |  | When the order was handed to the carrier |
| order_delivered_customer_date | DATETIME |  | When the customer received the order |
| order_estimated_delivery_date | DATETIME |  | Estimated delivery date shown to the customer |
| customer_unique_id | TEXT |  | Unique ID of the real customer across orders |
| customer_state | TEXT |  | Customer state (2-letter code) |
| order_value | FLOAT |  | Total payment value of the order (BRL) |
| payment_count | FLOAT |  | Number of payments on the order |
| max_installments | FLOAT |  | Highest installment count on the order |
| items_total | FLOAT |  | Sum of item prices on the order |
| freight_total | FLOAT |  | Sum of freight on the order |
| item_count | FLOAT |  | Number of items in the order |
| main_payment_type | TEXT |  | Main payment method of the order |
| review_score | FLOAT |  | Customer rating from 1 to 5 |
| delivery_days | FLOAT |  | Days from purchase to delivery |
| delivery_delay_days | FLOAT |  | Days between actual and estimated delivery |
| is_delayed | FLOAT |  | 1 if delivered after the estimated date, else 0 |
| delivery_status | TEXT |  | on_time, delayed or not_delivered |
| purchase_month | TEXT |  | Purchase month (year-month) |
| purchase_year | INTEGER |  | Purchase year |
| purchase_weekday | TEXT |  | Day of the week of the purchase |

### `customer_features`

Engineered customer-level features (notebook 05).

| Column | Type | Key | Description |
|---|---|---|---|
| customer_unique_id | TEXT |  | Unique ID of the real customer across orders |
| customer_order_count | BIGINT |  | Number of orders by the customer |
| customer_total_spending | FLOAT |  | Total spending by the customer (BRL) |
| customer_avg_order_value | FLOAT |  | Average order value of the customer (BRL) |
| first_order_date | DATETIME |  | Date of the customer's first order |
| last_order_date | DATETIME |  | Date of the customer's latest order |
| is_repeat_customer | BIGINT |  | 1 if the customer ordered more than once, else 0 |
| customer_state | TEXT |  | Customer state (2-letter code) |

### `seller_features`

Engineered seller-level features (notebook 05).

| Column | Type | Key | Description |
|---|---|---|---|
| seller_id | TEXT |  | Unique seller ID |
| seller_revenue | FLOAT |  | Total revenue of the seller (BRL) |
| seller_order_count | BIGINT |  | Number of orders served by the seller |
| seller_items_sold | BIGINT |  | Number of items sold by the seller |
| seller_avg_review | FLOAT |  | Average review score of the seller's orders |
| seller_avg_delivery_days | FLOAT |  | Average delivery time of the seller's orders |
| seller_delay_rate | FLOAT |  | Share of the seller's orders delivered late |
| seller_city | TEXT |  | Seller city |
| seller_state | TEXT |  | Seller state (2-letter code) |

### `item_features`

Engineered item-level features (notebook 05).

| Column | Type | Key | Description |
|---|---|---|---|
| order_id | TEXT |  | Unique order ID |
| order_item_id | BIGINT |  | Item sequence number within the order |
| product_id | TEXT |  | Unique product ID |
| seller_id | TEXT |  | Unique seller ID |
| shipping_limit_date | TEXT |  | Deadline for the seller to hand the item to the carrier |
| price | FLOAT |  | Item price (BRL) |
| freight_value | FLOAT |  | Freight cost for the item (BRL) |
| product_category_name | TEXT |  | Product category (Portuguese) |
| product_category_name_english | TEXT |  | Product category (English) |
| category | TEXT |  | Product category used for analysis (unknown if missing) |
| item_total | FLOAT |  | Item price plus freight (BRL) |

## ER Diagram

![ER diagram](../docs/er_diagram.png)

**Relationships**

- One customer places many orders (`orders.customer_id` -> `customers.customer_id`).
- One order contains many items (`items.order_id` -> `orders.order_id`).
- Each item links to one product and one seller.
- One order has one or more payments and one review (after cleaning).
- Products link to the English category names through `category_translation`.
- `geolocation` joins customers and sellers by zip code prefix. This link is logical only, not an enforced foreign key, because not every prefix exists in the geolocation data.
